# Corporate Events 2019–2026: M&A, Layoffs, IPOs: starter notebook

This file is a **sample**. The full data, the withheld fields and monthly updates come from the [Radar API](https://github.com/fliarbi/radar-api-mcp). This notebook loads the sample, explores it, then gets one record's full detail from the API.

In [ ]:
import glob, pandas as pd, matplotlib.pyplot as plt
path = (glob.glob('/kaggle/input/**/corporate_events_2019_2026_sample.csv', recursive=True) or ['corporate_events_2019_2026_sample.csv'])[0]
df = pd.read_csv(path, low_memory=False)
print(df.shape); df.head()

In [ ]:
df.describe(include="all").T.head(30)

In [ ]:
df['event_date'] = pd.to_datetime(df.event_date)
t = df.groupby([df.event_date.dt.year, 'event_type']).size().unstack(fill_value=0)
t.plot.bar(stacked=True, figsize=(10, 4), title='Event reports by year and type (sample)')
t

In [ ]:
df[df.event_type == 'Layoffs'].groupby('sector').size().sort_values(ascending=False).head(15)

In [ ]:
df[(df.event_type == 'M&A') & (df.role == 'acquirer')].company_name.value_counts().head(20)

## Get the full record from the Radar API

The sample holds the event sentence and source. The API returns the full timeline for a company, including leadership, earnings and funding events, plus momentum and risk summaries. [Create a key](https://fliarbi.com/radar-api/developer), then add it to Kaggle **Add-ons → Secrets** as `RADAR_API_KEY`.

In [ ]:
import os, json, requests
key = os.environ.get('RADAR_API_KEY')
try:
    from kaggle_secrets import UserSecretsClient
    key = key or UserSecretsClient().get_secret('RADAR_API_KEY')
except Exception:
    pass
value = df['company_domain'].dropna().astype(str).iloc[0]
if key:
    r = requests.post('https://radar-api.fliarbi.com/v1/companies/events', headers={'Authorization': f'Bearer {key}'},
                      json={'domain': value}, timeout=60)
    print(json.dumps(r.json(), indent=1)[:3000])
else:
    print('Set RADAR_API_KEY to fetch the full record for', value)

Each row also has `fliar_profile_url`, the company's full profile. Repository, docs and MCP setup: https://github.com/fliarbi/radar-api-mcp